In [1]:
# Tools in LlamaIndex
#
# Creating a FunctionTool
#
# Tools in LlamaIndexLet's create a basic `FunctionTool` and call it.
#

from llama_index.core.tools import FunctionTool

def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"

tool = FunctionTool.from_defaults(
    get_weather,
    name="my_weather_tool",
    description="Useful for getting the weather for a given location.",
)
tool.call("New York")

Getting weather for New York


ToolOutput(content='The weather in New York is sunny', tool_name='my_weather_tool', raw_input={'args': ('New York',), 'kwargs': {}}, raw_output='The weather in New York is sunny', is_error=False)

In [2]:
# Creating a QueryEngineTool
#
# Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. 
#

import chromadb

from llama_index.core import VectorStoreIndex
#from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
#from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

#
# NOTE: vector_store here relies on components notebook having populated alfred_chroma_db
#

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

#
# using Ollama embedding 
#

from llama_index.embeddings.ollama import OllamaEmbedding

BGE_SMALL_EN_V15_Q4_K_M = "qllama/bge-small-en-v1.5:q4_k_m"

MODEL = BGE_SMALL_EN_V15_Q4_K_M

embed_model = OllamaEmbedding(
    model_name=MODEL,
    base_url="http://localhost:11434",
    ollama_additional_kwargs={"mirostat": 0},
)
#embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

#
# using Ollama LLM
#

from llama_index.llms.ollama import Ollama

QWEN25_CODER_14B_Q4_K_M = "qwen2.5-coder:14b"

MODEL = QWEN25_CODER_14B_Q4_K_M

llm = llm = Ollama(
    base_url="http://localhost:11434",
    model=MODEL,
    timeout=0
)
#llm = HuggingFaceInferenceAPI(model_name="meta-llama/Llama-3.2-3B-Instruct")

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="some useful name",
    description="some useful description",
)

import nest_asyncio
nest_asyncio.apply()  # This is needed to run the query engine

await tool.acall(
    "Responds about research on the impact of AI on the future of work and society?"
)

ToolOutput(content='The provided context does not offer any information regarding a response to research on the impact of AI on the future of work and society. The focus appears to be on a social justice educator or activist working with diversity, equity, and inclusion issues, rather than on technology or AI impacts.', tool_name='some useful name', raw_input={'input': 'Responds about research on the impact of AI on the future of work and society?'}, raw_output=Response(response='The provided context does not offer any information regarding a response to research on the impact of AI on the future of work and society. The focus appears to be on a social justice educator or activist working with diversity, equity, and inclusion issues, rather than on technology or AI impacts.', source_nodes=[NodeWithScore(node=TextNode(id_='a47dc3d0-e366-43e3-b57e-bb2e37b957a0', embedding=None, metadata={'file_path': '/home/mikejay/Documents/github.com.d/brown-coat/agents-course/notebooks/unit2/llama-ind

In [3]:
# Creating Toolspecs
#
# Let's create a `ToolSpec` from the `GmailToolSpec` from the LlamaHub and convert it to a list of tools. 
#

from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec()
tool_spec_list = tool_spec.to_tool_list()
tool_spec_list

In [4]:
# To get a more detailed view of the tools, we can take a look at the `metadata` of each tool.
#

[print(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]

load_data load_data() -> List[llama_index.core.schema.Document]
Load emails from the user's account.
search_messages search_messages(query: str, max_results: Optional[int] = None)

        Searches email messages given a query string and the maximum number
        of results requested by the user
           Returns: List of relevant message objects up to the maximum number of results.

        Args:
            query (str): The user's query
            max_results (Optional[int]): The maximum number of search results
            to return.

        
create_draft create_draft(to: Optional[List[str]] = None, subject: Optional[str] = None, message: Optional[str] = None) -> str

        Create and insert a draft email.
           Print the returned draft's message and id.
           Returns: Draft object, including draft id and message meta data.

        Args:
            to (Optional[str]): The email addresses to send the message to
            subject (Optional[str]): The subject for th

[None, None, None, None, None, None]